In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from marine_qc import (
    Climatology,
    do_bayesian_buddy_check,
)

In [ ]:
from data import get_climatology_data, get_grouped_data

# How to use quality control checks with grouped reports

The ``marine_qc`` toolbox provides some quality control (QC) checks that work a group of marine reports. Creating some test dataset we can show how to use them on "real" data. There is one check that is shown here:

* a buddy check that tests that each observation is reasonably close to the average of its near neighbours in time and space

For more information of all available QC checks on a group of individual reports see [the overview of QC functions for grouped reports](https://marine-qc.readthedocs.io/en/stable/overview_grp.html).

The test dataset we provide is a ``pandas.DataFrame`` including marine reports representing the track of five ships. The QC functions return a QC flag for each individual report. The flags are:

* `0`: the check has passed
* `1`: the check has failed
* `2`: the check is untestable

In [ ]:
data = get_grouped_data()
data

The ship data include six different parameters (`lat`, `lon`, `date`, and `sst`).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

colors = plt.cm.tab10.colors

i = 0
for color, (platform, group) in zip(colors, data.groupby("platform"), strict=False):
    ax1.plot(
        group["lon"],
        group["lat"],
        "-o",
        ms=5,
        label=platform,
        color=color,
    )

    ax2.plot(
        group["date"],
        group["sst"],
        "-o",
        ms=5,
        label=platform,
        color=color,
    )

    outlier = data.loc[(data.platform == platform) & (data.date == pd.Timestamp(f"2026-07-01 {13 + i}:00"))]

    ax1.scatter(
        outlier.lon,
        outlier.lat,
        s=150,
        facecolors="none",
        edgecolors="red",
        linewidth=2,
        zorder=10,
    )

    ax2.scatter(
        outlier.date,
        outlier.sst,
        s=150,
        facecolors="none",
        edgecolors="red",
        linewidth=2,
        zorder=10,
    )

    i += 1

ax1.set_xlabel("Longitude (°)")
ax1.set_ylabel("Latitude (°)")
ax1.set_title("Buddy observations")
ax1.legend(loc="best")

ax2.set_xlabel("Time")
ax2.set_ylabel("SST (°C)")
ax2.set_title("Sea surface temperature")
ax2.tick_params(axis="x", rotation=30)

plt.show()

We see five different ships. Each of them includes one value that is much higher than the others. These values should be detected and flagged as `1` (failed).

Firtly, we import some external data. This is a `xarray.Dataset` containing [CF](https://cfconventions.org/)-compliant data:

* a land-sea mask where `1` denotes a land point and `0` denotes a sea point (`land_sea_mask`)
* a simple, latitudinally and longitudinally dependent sea-surface temperature field (`sst`)
* a sea-surface temperature standard deviation of grid cell minus neighbourhood mean (`sst_stde1`)
* a sea-surface temperature standard deviation of point observation minus grid cell mean (`sst_stde2`)
* a sea-surface temperature standard deviation of neighbourhood mean uncertainty (`sst_stde3`)

In [ ]:
climatology_data = get_climatology_data()
climatology_data

For the buddy check, we need some external data:

* `climatology`: The climatological average(s) used to calculate anomalies.
* `stdev1`: Field of standard deviations representing standard deviation of difference between target gridcell and complete neighbour average (grid area to neighbourhood difference)
* `stdev2`: Field of standard deviations representing standard deviation of difference between a single observation and the target gridcell average (point to grid area difference)
* `stdev3`: Field of standard deviations representing standard deviation of difference between random neighbour gridcell and full neighbour average (uncertainty in neighbour average)

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 8),
    constrained_layout=True,
)

plots = [
    ("sst", "sea-surface temperature", "RdYlBu_r"),
    ("sst_stdev1", "stdev1", "viridis"),
    ("sst_stdev2", "stdev2", "viridis"),
    ("sst_stdev3", "stdev3", "viridis"),
]

for ax, (var, title, cmap) in zip(axes.flat, plots, strict=False):
    climatology_data[var].isel(time=0).plot(
        ax=ax,
        cmap=cmap,
        add_colorbar=True,
    )
    ax.set_title(title)

plt.show()

Unfortunately, the standard deviations have to be converted manually to `marine_qc.Climatology` classes.

In [ ]:
sst_stdev1 = Climatology(climatology_data.sst_stdev1)
sst_stdev2 = Climatology(climatology_data.sst_stdev2)
sst_stdev3 = Climatology(climatology_data.sst_stdev3)

Besiedes the external climatologies some additional parameters needs to be set:

* `prior_probability_of_gross_error`: prior probability of gross error, which is the background rate of gross errors (`0.05`)
* `quantization_interval`: smallest possible increment in the input values (`0.1`)
* `one_sigma_measurement_uncertainty`: estimated one sigma measurement uncertainty (`1.0`)
* `limits`: list with three members which specify the search range for the buddy check (`[2, 2, 4]`)
* `noise_scaling`: tuning parameter used to multiply `stdev2` (`3.0`)
* `maximum_anomaly`: largest absolute anomaly, assumes that the maximum and minimum anomalies have the same magnitude (`8.0`)
* `fail_probability`: probability of gross error that corresponds to a failed test. Anything with a probability of gross error greater than fail_probability will be considered failing (`0.3`)

In [ ]:
qc_buddy = do_bayesian_buddy_check(
    value=data.sst,
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    climatology=climatology_data.sst,
    stdev1=sst_stdev1,
    stdev2=sst_stdev2,
    stdev3=sst_stdev3,
    prior_probability_of_gross_error=0.05,
    quantization_interval=0.1,
    one_sigma_measurement_uncertainty=1.0,
    limits=[2, 2, 4],
    noise_scaling=3.0,
    maximum_anomaly=8.0,
    fail_probability=0.3,
)
pd.DataFrame({**data, "qc_flag": qc_buddy})

We get the expected results.